# Source Analysis
Let's show insights about the scraped news

In [ ]:
import pandas as pd
import plotly.express as px

data = pd.read_csv("news.csv")
data.rename(columns={"Date scraped": "Date"}, inplace=True)
data["Sentiment_label"] = data["Sentiment"].apply(lambda x: "Positive" if x > 0.05  else "Negative" if x < -0.05 else "Neutral")
data["Date"] = data["Date"].apply(lambda x: pd.to_datetime(x).date())

In [ ]:
data.head()

## I. Insights Per Day

#### 1. Proportion of topics

In [ ]:
tab_01 = data[["Date", "Topics"]].value_counts().reset_index()
fig_01 = px.bar(tab_01,
                x="Date",
                y="count",
                color="Topics", color_discrete_sequence=px.colors.qualitative.Bold,
                opacity=0.8,
                title="Proportion of Topics Per Day",
                )
fig_01.update_layout(
    title_font=dict(size=20, weight="bold"),
    title_x=0.5,
    yaxis=dict(zerolinecolor="grey", gridcolor="white"),
    font=dict(color="black", family="times new roman"),
    plot_bgcolor="lightgrey",
    paper_bgcolor="lightgrey",
)


fig_01.show()

#### 2. Number of articles

In [ ]:
tab_02 = data.groupby("Date")["body"].agg(count="count")
fig_02 = px.bar(tab_02,
                color_discrete_sequence=["teal"],
                y="count",
                opacity=0.8,
                title="Number of article per day")
fig_02.update_layout(
    title_font=dict(size=30, weight="bold"),
    title_x=0.5,
    yaxis=dict(zerolinecolor="lightgrey", gridcolor="rgba(255,255,255,0.3)", gridwidth=2),
    xaxis=dict(gridcolor="rgba(255,255,255,0.3)", gridwidth=2, linecolor="teal", linewidth=2),
    plot_bgcolor="lightgrey",
    paper_bgcolor="lightgrey",
    font=dict(color="black", family="times new roman"),
)
fig_02.show()

#### 3. Number of companies mentioned

In [ ]:
tab_03 = data.copy()[["Date", "Org"]]
tab_03["Company_count"] = tab_03["Org"].apply(lambda x: len(x.split(", ")) if len(x) > 2 else 0)
tab_03 = tab_03.groupby("Date")["Company_count"].sum().reset_index()
fig_03 = px.bar(tab_03, x="Date",
                      y="Company_count",
                      color_discrete_sequence=["#07090F"],
                      title="Number of Companies Mentioned per day",
                     text_auto=True,
                      )
fig_03.update_layout(
    title_font=dict(size=30, weight="bold", color="black"),
    title_x=0.5,
    yaxis=dict(zerolinecolor="lightgrey", showticklabels=False),
    plot_bgcolor="lightgrey",
    paper_bgcolor="lightgrey",
    font=dict(color="black", family="times new roman"),
)
fig_03.show()

#### 4. Proportion of sentiment

In [ ]:
tab_04 = data[["Date", "Sentiment_label"]].value_counts().reset_index()
fig_04 = px.bar(tab_04, x="Date",
                y="count",
                color="Sentiment_label",
                color_discrete_sequence=["cyan", "crimson", "dimgrey"],
                title="Proportion of Sentiments per Day",
                opacity=0.8,
                labels={"Sentiment_label": "Sentiment"},
                )
fig_04.update_layout(
    title_font=dict(size=30, weight="bold", color="black"),
    title_x=0.5,
    yaxis=dict(zerolinecolor="lightgrey", gridcolor="gray"),
    xaxis=dict(linecolor="black", linewidth=1),
    plot_bgcolor="lightgrey",
    paper_bgcolor="lightgrey",
    font=dict(color="black", family="times new roman"),
    legend_title="Sentiments"
)
fig_04.show()

#### 5. Average Scandal Intensity

In [ ]:
tab_05 = data.groupby("Date")["Scandal_distance"].agg(avg_intensity="mean").reset_index()
fig_05 = px.line(tab_05,
                 x="Date",
                 y="avg_intensity",
                 markers=True,
                 color_discrete_sequence=["green"],
                 title="Average Scandal Distance Intensity per Day",)
fig_05.update_layout(
    title_font=dict(size=30, weight="bold"),
    title_x=0.5,
    yaxis=dict(zerolinecolor="lightgrey", linecolor="black", linewidth=2),
    xaxis=dict(linecolor="black", linewidth=2),
    plot_bgcolor="lightgray",
    paper_bgcolor="lightgray",
    font=dict(size=14, color="black", family="times new roman"),
)
fig_05.show()

## II. Overall Insights

#### 1. Proportion of topic per sentiment

In [ ]:
tab_06 = data[["Sentiment_label", "Topics"]].value_counts().reset_index()

fig_06 = px.sunburst(tab_06,
                     path=["Sentiment_label", "Topics"],
                     values="count",
                     title="Proportion Of Topics Per Sentiments Per Day",
                     color="Sentiment_label",
                     color_discrete_sequence=px.colors.qualitative.Prism,
                     labels={"Sentiment_label": "Sentiment"},
                     )
fig_06.update_layout(
    title_font=dict(size=25, weight="bold"),
    title_x=0.5,
    font=dict(size=16, color="black", family="times new roman"),
    paper_bgcolor="lightgray",
    width=900,
    height=500,
    margin=dict(t=50, l=5, r=5, b=25)
)
fig_06.show()

#### 2. Most mentioned ORG

In [ ]:
tab_07_filter = data.copy()[["Date", "Org"]]
tab_07_filter["Org"] = tab_07_filter["Org"].apply(lambda x: list(x.replace("[", "").replace("]", "").split(", ")) if len(x) > 2 else [])
tab_07_filter = tab_07_filter.explode("Org")
tab_07_filter.fillna("empty", inplace=True)
tab_07 = (tab_07_filter.groupby("Org")["Org"]
                       .agg(count="count")
                       .sort_values(by="count", ascending=False)
                       .reset_index()
                       .drop([0])
                        .head(6))

fig_07 = px.bar(tab_07,
                x="count",
                y="Org",
                color="Org",
                color_discrete_sequence=px.colors.qualitative.Dark2,
                title="Most mentioned ORG",
                )
fig_07.update_layout(
    title_font=dict(size=30, weight="bold"),
    title_x=0.5,
    yaxis=dict(showticklabels=False, linewidth=1, color="lightgrey"),
    yaxis_title=None,
   xaxis=dict(gridcolor="gray", gridwidth=2, linecolor="gray", linewidth=1, griddash="dot"),
    plot_bgcolor="lightgray",
    paper_bgcolor="lightgray",
    font=dict(size=14, color="black", family="times new roman"),
    barcornerradius=5,
    legend_title_text="ORG",
    legend=dict(
        bgcolor='rgba(255,255,255,0.2)',
    )
)
fig_07.show()